# MNIST: van data begrijpen naar je eerste neurale netwerk

In deze opdracht doorloop je een kleine maar complete machine-learningworkflow met MNIST:

**data begrijpen → EDA → train/test split → preprocessing → baseline → neural network → evaluatie**

Je hoeft nog niet alle Python-syntax uit je hoofd te kennen. Bij nieuwe pandas- en Keras-stappen krijg je daarom steeds:

- de functie of methode die je nodig hebt;
- een korte uitleg van wat die functie doet;
- een code-skelet of gerichte hint.

Daarna schrijf je zelf de relevante regel code.

De uitgebreidere visualisatiecode met matplotlib en seaborn staat meestal al voor je klaar. Het doel is dat je zowel leert programmeren als begrijpt waarom iedere stap nodig is.

## Leerdoelen

Na deze opdracht kun je:

- een pandas `DataFrame` inspecteren met veelgebruikte functies;
- labels tellen, data selecteren, groeperen en eenvoudige nieuwe variabelen maken;
- relevante EDA-resultaten interpreteren;
- uitleggen waarom train-, validation- en testdata verschillende functies hebben;
- preprocessingkeuzes voor MNIST onderbouwen;
- een eenvoudige baseline bepalen;
- met Keras zelf een eenvoudig Dense neural network opbouwen, compileren en trainen;
- learning curves, test accuracy en een confusion matrix interpreteren;
- uitleggen waarom goede prestaties op MNIST begrensd bewijs zijn voor een echte toepassing.

# 1. Setup

Voer deze cellen uit. Dit is alleen de technische voorbereiding; deze code hoef je niet zelf te schrijven.

In [ ]:
%pip install -q pandas matplotlib seaborn scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 20)
np.random.seed(42)

print("Libraries geladen.")

# 2. MNIST laden

MNIST bestaat uit afbeeldingen van handgeschreven cijfers van 0 t/m 9.

Iedere afbeelding is **28 × 28 pixels = 784 pixelwaarden**. In deze dataset is iedere afbeelding platgeslagen tot één rij.

- 1 rij = 1 afbeelding
- 784 pixelkolommen = features
- `label` = het juiste cijfer

De dataset wordt hieronder al volledig voor je geladen. Je hoeft het ophalen en omzetten van MNIST dus niet zelf te programmeren.

In [ ]:
mnist = fetch_openml("mnist_784", version=1, as_frame=True)

X = mnist.data.astype("uint8")
y = mnist.target.astype("int8")

pixel_cols = X.columns.tolist()

df = X.copy()
df["label"] = y.to_numpy()

print("Dataset geladen.")

## 2.1 Eerste pandas-stappen

Een pandas `DataFrame` is een tabel. Drie functies die je vaak gebruikt:

```python
df.shape
df.head()
df["kolomnaam"].value_counts()
```

Schrijf hieronder zelf drie regels waarmee je:

1. de vorm van `df` bekijkt;
2. de eerste vijf rijen toont;
3. telt hoe vaak ieder label voorkomt.

In [ ]:
# Schrijf hieronder je drie pandas-regels.

# Hint 1: gebruik df.shape
# Hint 2: gebruik df.head()
# Hint 3: selecteer eerst de kolom "label" en gebruik daarna .value_counts()

**Korte vraag:** waarom heeft `df` 785 kolommen terwijl een MNIST-afbeelding maar 784 pixels heeft?

>

# 3. Sanity check

Voordat we grafieken maken, controleren we of de data technisch logisch is.

Gebruik hiervoor de volgende pandas-methodes:

```python
.unique()                  # unieke waarden
.isna().sum().sum()        # totaal aantal ontbrekende waarden
.min()                     # minimum
.max()                     # maximum
```

Bij de pixelwaarden moet je eerst alleen de pixelkolommen selecteren met `df[pixel_cols]`.

Vul de vijf variabelen hieronder zelf in.

In [ ]:
# Vul de rechterkant van iedere regel in.

dataset_shape = ...       # gebruik df.shape
labels = ...              # gebruik .unique() op de labelkolom
missing_values = ...      # gebruik .isna().sum().sum()
min_pixel = ...           # selecteer pixel_cols en gebruik twee keer .min()
max_pixel = ...           # selecteer pixel_cols en gebruik twee keer .max()

print("Vorm:", dataset_shape)
print("Labels:", sorted(labels))
print("Ontbrekende waarden:", missing_values)
print("Pixelrange:", min_pixel, "t/m", max_pixel)

## 3.1 Zelf een steekproef nemen

Met:

```python
DataFrame.sample(n=..., random_state=...)
```

kun je willekeurige rijen bekijken.

Schrijf één regel waarmee je **vijf** willekeurige rijen uit `df` toont. Gebruik `random_state=42`.

In [ ]:
# Schrijf hier je regel met df.sample(...)

# 4. Train- en testdata maken

De testset simulereert data die we tijdens modelontwikkeling nog niet kennen. Daarom zetten we deze vroeg apart.

Gebruik `train_test_split()` met:

- `test_size=0.20`
- `random_state=42`
- `stratify=y_all`

De features en target zijn al voor je geselecteerd. Maak daarna zelf de split af.

In [ ]:
X_all = df[pixel_cols]
y_all = df["label"]

# Vul de drie argumenten met ... in.
X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=...,
    random_state=...,
    stratify=...
)

print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)

## 4.1 Controleer de klasseverdeling

Met:

```python
Series.value_counts(normalize=True)
```

bereken je verhoudingen in plaats van aantallen. Met `.sort_index()` zet je de labels daarna weer netjes van 0 t/m 9.

Schrijf zelf de twee regels voor train en test. De tabel eronder staat al klaar.

In [ ]:
train_percentage = ...   # gebruik y_train.value_counts(normalize=True).sort_index()
test_percentage = ...    # hetzelfde voor y_test

split_check = pd.DataFrame({
    "train_%": (train_percentage * 100).round(2),
    "test_%": (test_percentage * 100).round(2)
})

display(split_check)

**Korte vraag:** wat laat deze tabel zien over het effect van `stratify`?

>

# 5. EDA op de trainingsdata

Vanaf nu doen we de inhoudelijke EDA op de trainingsdata.

Maak eerst zelf `train_df`:

1. maak een kopie van `X_train` met `.copy()`;
2. voeg daarna `y_train` toe als kolom `"label"`.

Hint voor stap 2:

```python
train_df["label"] = y_train.to_numpy()
```

In [ ]:
# Stap 1: maak een kopie van X_train
train_df = ...

# Stap 2: voeg de labelkolom toe
...

display(train_df.head())

## 5.1 Hoe zijn de cijfers verdeeld?

Gebruik op de kolom `label`:

```python
.value_counts()
.sort_index()
```

om het aantal voorbeelden per cijfer te tellen.

Schrijf alleen de pandas-regel. De seaborn-visualisatie staat al voor je klaar.

In [ ]:
label_counts = ...

display(label_counts)

plt.figure(figsize=(9, 4))
sns.barplot(x=label_counts.index, y=label_counts.values)
plt.title("Aantal trainingsvoorbeelden per cijfer")
plt.xlabel("Cijfer")
plt.ylabel("Aantal afbeeldingen")
plt.show()

Gebruik vervolgens:

```python
.idxmax()
.idxmin()
```

om te vinden welk label het vaakst en minst vaak voorkomt.

In [ ]:
most_common_label = ...
least_common_label = ...

print("Meest voorkomende label:", most_common_label)
print("Minst voorkomende label:", least_common_label)

**Interpretatie:** is de class imbalance hier groot genoeg om direct oversampling nodig te maken?

>

## 5.2 Bekijk echte afbeeldingen

Bij beelddata hoort ook visuele inspectie.

Kies zelf eerst tien willekeurige rijen met `.sample()`. De matplotlib-code zet de 784 pixels daarna terug naar 28×28.

In [ ]:
# Schrijf zelf deze pandas-regel.
sample = ...   # neem 10 rijen uit train_df, random_state=42

fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    image = row[pixel_cols].to_numpy(dtype=np.uint8).reshape(28, 28)
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Label: {int(row['label'])}")
    ax.axis("off")

plt.tight_layout()
plt.show()

**Korte observatie:** noem één eigenschap die je bij meerdere afbeeldingen terugziet en die relevant kan zijn voor een model.

>

# 6. Van pixels naar eenvoudige EDA-features

784 losse pixels zijn lastig te interpreteren. Daarom maken we drie samenvattende variabelen per afbeelding.

Gebruik:

```python
X_train.mean(axis=1)
(condition).sum(axis=1)
```

Maak zelf:

- `mean_intensity`: gemiddelde pixelwaarde;
- `active_pixels`: aantal pixels groter dan 0;
- `bright_pixels`: aantal pixels groter dan of gelijk aan 200.

De eerste twee regels om `train_eda` te maken staan klaar.

In [ ]:
train_eda = pd.DataFrame(index=X_train.index)
train_eda["label"] = y_train

# Schrijf de drie pandas-regels zelf.
train_eda["mean_intensity"] = ...
train_eda["active_pixels"] = ...
train_eda["bright_pixels"] = ...

display(train_eda.head())

## 6.1 Actieve pixels per cijfer

De seaborn-code staat al klaar. Je hoeft de syntax van `sns.boxplot()` nog niet uit je hoofd te kennen.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=train_eda,
    x="label",
    y="active_pixels",
    showfliers=False
)
plt.title("Aantal actieve pixels per cijfer")
plt.xlabel("Cijfer")
plt.ylabel("Aantal pixels > 0")
plt.show()

## 6.2 Eenvoudig groeperen met pandas

Met:

```python
DataFrame.groupby("label")["kolom"].mean()
```

kun je per label een gemiddelde berekenen.

Bereken zelf het gemiddelde aantal `active_pixels` per label.

In [ ]:
mean_active_by_label = ...

display(mean_active_by_label.round(1))

print("Minste gemiddeld actieve pixels:", mean_active_by_label.idxmin())
print("Meeste gemiddeld actieve pixels:", mean_active_by_label.idxmax())

## 6.3 Het gemiddelde cijfer per klasse

Maak eerst met pandas één tabel waarin voor ieder label de gemiddelde waarde van alle pixelkolommen staat.

Gebruik:

```python
train_df.groupby("label")[pixel_cols].mean()
```

Schrijf die regel zelf. De visualisatie staat daarna klaar.

In [ ]:
mean_digits = ...

fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for label, ax in zip(range(10), axes.flat):
    avg_image = mean_digits.loc[label].to_numpy().reshape(28, 28)
    ax.imshow(avg_image, cmap="gray")
    ax.set_title(f"Gemiddelde {label}")
    ax.axis("off")

plt.tight_layout()
plt.show()

# 7. Datakwaliteit controleren

Gebruik pandas om vier controles zelf uit te voeren.

Je hebt hiervoor nodig:

```python
.isna().sum().sum()
.min().min()
.max().max()
.duplicated().sum()
```

Vul de variabelen in.

In [ ]:
missing_pixels = ...
min_pixel = ...
max_pixel = ...
duplicate_images = ...

print("Ontbrekende pixelwaarden:", missing_pixels)
print("Pixelrange:", min_pixel, "t/m", max_pixel)
print("Exacte dubbele afbeeldingen in train:", duplicate_images)

# 8. Preprocessing kiezen

Gebruik je EDA-resultaten en vul alleen de kolom **Nodig?** in.

| Mogelijke stap | Nodig? | Aanwijzing |
|---|---|---|
| Missing values imputeren |  | Kijk naar je missing-value check |
| Pixels normaliseren naar 0–1 |  | Pixels liggen nu tussen 0 en 255 |
| Oversampling |  | Kijk naar de class distribution |
| One-hot encoding van pixels |  | Pixels zijn numerieke intensiteiten |
| Data augmentation |  | Kan nuttig zijn, maar is niet noodzakelijk voor deze eerste benchmark |

## 8.1 Normaliseren

Zet nu zelf de train- en testfeatures om naar `float32` en deel alle waarden door `255.0`.

Gebruik:

```python
DataFrame.astype("float32")
```

Maak twee nieuwe variabelen: `X_train_scaled` en `X_test_scaled`.

In [ ]:
X_train_scaled = ...
X_test_scaled = ...

print("Train min:", X_train_scaled.min().min())
print("Train max:", X_train_scaled.max().max())
print("Datatype:", X_train_scaled.dtypes.iloc[0])

**Korte vraag:** waarom mag dezelfde vaste deling door 255 op train én test worden toegepast, terwijl je bijvoorbeeld een `StandardScaler` niet op de volledige dataset zou fitten?

>

# 9. EDA-conclusie vóór modelleren

Vul de drie zinnen aan met je eigen resultaten.

- De dataset bevat ...
- Daarom voeren we als preprocessing ...
- Voor een echte toepassing buiten MNIST is een belangrijk risico ...

# 10. Van EDA naar je eerste neural network

We vervolgen dezelfde workflow:

**validation → baseline → model → training → generalization → test → foutenanalyse**

De input bestaat uit 784 genormaliseerde pixelwaarden. Het target is één cijfer van 0 t/m 9.

## 10.1 Maak een validation set

We gebruiken opnieuw `train_test_split()`, maar nu alleen binnen onze trainingsdata.

Gebruik:

- `test_size=0.20`
- `random_state=42`
- `stratify=y_train`

Schrijf de split zelf.

In [ ]:
X_model_train, X_val, y_model_train, y_val = train_test_split(
    X_train_scaled,
    y_train,
    test_size=...,
    random_state=...,
    stratify=...
)

print("Model train:", X_model_train.shape)
print("Validation: ", X_val.shape)

Keras werkt gemakkelijk met NumPy-arrays. Gebruik op iedere DataFrame of Series:

```python
.to_numpy()
```

Zet zelf train, validation en test om.

In [ ]:
train_images = ...
val_images = ...
test_images = ...

train_labels = ...
val_labels = ...
test_labels = ...

print(train_images.shape, val_images.shape, test_images.shape)

# 11. Eerst een baseline

De random baseline is bij tien ongeveer even waarschijnlijke klassen ongeveer 10%.

Bereken zelf de majority baseline met:

```python
y_model_train.value_counts(normalize=True).max()
```

In [ ]:
random_baseline = 1 / 10
majority_baseline = ...

print(f"Random baseline:   {random_baseline:.2%}")
print(f"Majority baseline: {majority_baseline:.2%}")

# 12. Bouw het neural network

Gebruik Keras om dit netwerk zelf op te bouwen:

```text
784 inputs
    ↓
Dense(512) + ReLU
    ↓
Dense(10) + Softmax
```

De structuur van `Sequential` staat hieronder. Vul de getallen en activatiefuncties zelf in.

Gebruik:

```python
keras.Input(shape=(...,))
layers.Dense(..., activation="...")
```

In [ ]:
# Voer deze installatie alleen uit als Keras/TensorFlow nog niet beschikbaar is.
# %pip install -q keras tensorflow

import keras
from keras import layers

keras.utils.set_random_seed(42)

model = keras.Sequential([
    keras.Input(shape=(...,)),
    layers.Dense(..., activation="..."),
    layers.Dense(..., activation="...")
])

model.summary()

**Korte vraag:** waarom heeft de laatste laag precies 10 units?

>

# 13. Compile het model

Gebruik:

- optimizer: `"adam"`
- loss: `"sparse_categorical_crossentropy"`
- metric: `"accuracy"`

Schrijf zelf de `model.compile(...)`-aanroep.

In [ ]:
# Gebruik model.compile(optimizer=..., loss=..., metrics=[...])

# 14. Train het model

Gebruik `model.fit()` met:

- `train_images`
- `train_labels`
- `epochs=10`
- `batch_size=128`
- `validation_data=(val_images, val_labels)`

Bewaar het resultaat in een variabele `history`.

In [ ]:
# history = model.fit(...)

# 15. Learning curves

Keras bewaart de scores in `history.history`.

Maak daar eerst zelf een pandas DataFrame van met:

```python
pd.DataFrame(...)
```

De seaborn-code voor de curves staat daarna klaar.

In [ ]:
history_df = ...

history_df.index = history_df.index + 1
history_df.index.name = "epoch"

display(history_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(x=history_df.index, y=history_df["accuracy"], label="Train", ax=axes[0])
sns.lineplot(x=history_df.index, y=history_df["val_accuracy"], label="Validation", ax=axes[0])
axes[0].set_title("Accuracy per epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")

sns.lineplot(x=history_df.index, y=history_df["loss"], label="Train", ax=axes[1])
sns.lineplot(x=history_df.index, y=history_df["val_loss"], label="Validation", ax=axes[1])
axes[1].set_title("Loss per epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")

plt.tight_layout()
plt.show()

**Interpretatie:** zie je een moment waarop training nog verbetert, terwijl validation nauwelijks meer verbetert of slechter wordt?

>

# 16. Nu pas: de testset

Gebruik:

```python
model.evaluate(...)
```

om de uiteindelijke test loss en test accuracy te berekenen.

De functie geeft twee waarden terug. Sla die op als `test_loss` en `test_accuracy`.

In [ ]:
test_loss, test_accuracy = ...

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Majority baseline: {majority_baseline:.2%}")

# 17. Voorspellingen maken

Gebruik:

```python
model.predict(test_images)
```

om voor alle testafbeeldingen voorspellingen te maken.

Gebruik daarna `.argmax(axis=1)` om van de tien Softmax-scores per afbeelding één voorspeld label te maken.

In [ ]:
predictions = ...
y_pred = ...

print("Vorm van predictions:", predictions.shape)
print("Eerste 10 voorspelde labels:", y_pred[:10])

## 17.1 Bekijk één voorspelling

De visualisatiecode staat klaar. Verander `example_index` gerust naar een ander getal.

In [ ]:
example_index = 0

scores = predictions[example_index]
predicted_class = y_pred[example_index]
true_class = test_labels[example_index]

plt.imshow(test_images[example_index].reshape(28, 28), cmap="gray")
plt.title(f"True: {true_class} | Pred: {predicted_class}")
plt.axis("off")
plt.show()

print("Softmax-scores:", np.round(scores, 3))
print("Hoogste score:", round(float(scores.max()), 3))

# 18. Confusion matrix

De voorspellingen `y_pred` heb je zelf gemaakt. Nu gebruiken we die om een confusion matrix te berekenen en met seaborn te visualiseren.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_labels, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cbar=False)
plt.title("Confusion matrix op de testset")
plt.xlabel("Voorspeld label")
plt.ylabel("Werkelijk label")
plt.show()

**Korte observatie:** noem één combinatie van cijfers die het model relatief vaak met elkaar verwart.

>

# 19. Bekijk echte fouten

De code hieronder selecteert tien fout geclassificeerde voorbeelden. Je hoeft deze matplotlib-code niet zelf te schrijven.

In [ ]:
wrong_indices = np.where(y_pred != test_labels)[0]
chosen = wrong_indices[:10]

fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for ax, idx in zip(axes.flat, chosen):
    ax.imshow(test_images[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"True: {int(test_labels[idx])} | Pred: {int(y_pred[idx])}")
    ax.axis("off")

plt.tight_layout()
plt.show()

# 20. Eindconclusie

Schrijf ongeveer 5–6 zinnen waarin je de hele workflow samenvat. Verwerk daarin:

**wat je EDA liet zien → welke preprocessing je koos → waarom train/validation/test gescheiden zijn → hoe het neural network presteerde → welke beperking MNIST heeft voor een echte toepassing.**

>